# Part a: Connect 5 against the bots

Train a neural network that beats as many Connect 5 bots as possible.

**The game.** Connect 5 is Connect 4 on a bigger board: 9 columns, 9 rows. Players
take turns to drop a stone into a column; it falls to the lowest empty cell. The
first player with **5 in a row** (across, up or diagonal) wins. A full board is a
draw.

**What this notebook gives you:** the game, four bots to train and test against
(`random`, `easy`, `medium`, `hard`), a network template, a working PPO trainer
that plays the bots, an evaluation, a replay viewer, and the hand-in.

**How we grade:** your bot plays a set of **held-out bots** that you do not get:
different styles, and some much stronger than `hard`. Each opening is played
twice, once with each color. Win 1, draw 0.5, loss 0. Your score is the mean.

**Rules for your bot** (details in `PROBLEM.md`):
* at most **200 million parameters**, all in `agent.net`;
* its moves come from your network (a search that calls your network is allowed);
* at most **1 second per move** on one CPU core (a move over 2 seconds loses the game).

**Reading:** [Reinforcement learning fundamentals](https://medium.com/@812jugalgajjar/reinforcement-learning-fundamentals-understanding-the-framework-42963a7a5b40)

In [ ]:
import pathlib, zipfile
if not pathlib.Path("connectk").exists():          # first run: unpack the code next to this notebook
    zipfile.ZipFile("connect5.zip").extractall(".")

import copy, dataclasses, json, shutil, time
import numpy as np
import torch
from torch.distributions import Categorical

from connectk import CONNECT5, Board
from connectk.agents import MAX_PARAMS, check
from connectk.arena import RandomPlayer, make_openings, play_match
from connectk.packed import load_packed
from connectk.tournament import round_robin
from connectk.viewer import record_game, write_html

SPEC = CONNECT5
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
print(SPEC, "| training on", device)

## 1. The game

`Board` holds a position. `board.play(col)` drops a stone for the side to move.
Row 0 is the bottom row.

In [ ]:
b = Board(SPEC)
for col in [4, 4, 3, 5, 2, 3]:
    b.play(col)
print(b.render())
print("to move:", b.player, " legal columns:", b.legal_moves(), " winner:", b.winner)

## 2. The bots

Four bots, from weak to strong. Treat them as black boxes: play them and watch
them, but do not unpack `bots.bin`. Here they play each other (each pair plays
every opening twice, once with each color).

In [ ]:
bots = load_packed("bots.bin", "connect5_bots", seed=0)
table = round_robin(bots, SPEC, make_openings(SPEC, 8, seed=1), anchor=("medium", 1000.0))
print(table)

## 3. Your network

The next cell writes `model_a.py`: the network and its input encoding. Change it
as you like (more channels, more blocks, other inputs), then run the cell after
it to reload. The template is a small residual network.

In [ ]:
%%writefile model_a.py
"""Your network and its input encoding. Change anything you like.

The hand-in copies this file into your submission as model.py, so the
network you train is exactly the network we play. Keep it under 200 million
parameters, and fast enough for 1 second per move on one CPU core.
"""
import numpy as np
import torch
from torch import nn


def encode(boards):
    """Input for a list of connectk Boards: (B, 4, rows, cols), from the view of the side to move.

    planes: my stones, the opponent's stones, the landing cell of each column, all ones
    """
    g = np.stack([b.grid for b in boards])
    me = np.array([b.player for b in boards])[:, None, None]
    h = np.stack([b.heights for b in boards])
    rows = g.shape[1]
    x = np.zeros((len(boards), 4) + g.shape[1:], dtype=np.float32)
    x[:, 0] = g == me
    x[:, 1] = (g != 0) & (g != me)
    n, c = np.nonzero(h < rows)
    x[n, 2, h[n, c], c] = 1.0
    x[:, 3] = 1.0
    return torch.from_numpy(x)


class Block(nn.Module):
    """Residual block: conv, ReLU, conv, add the input, ReLU."""

    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        return torch.relu(x + self.c2(torch.relu(self.c1(x))))


class Net(nn.Module):
    """Outputs: one logit per column (the policy) and a value in [-1, 1] for the side to move."""

    def __init__(self, rows, cols, channels=64, blocks=4):
        super().__init__()
        self.stem = nn.Conv2d(4, channels, 3, padding=1)
        self.blocks = nn.Sequential(*[Block(channels) for _ in range(blocks)])
        self.policy = nn.Sequential(nn.Conv2d(channels, 2, 1), nn.ReLU(), nn.Flatten(), nn.Linear(2 * rows * cols, cols))
        self.value = nn.Sequential(nn.Conv2d(channels, 1, 1), nn.ReLU(), nn.Flatten(),
                                   nn.Linear(rows * cols, 64), nn.ReLU(), nn.Linear(64, 1), nn.Tanh())

    def forward(self, x):
        h = self.blocks(torch.relu(self.stem(x)))
        return self.policy(h), self.value(h).squeeze(-1)

In [ ]:
import importlib, model_a
importlib.reload(model_a)                     # pick up your edits to model_a.py
from model_a import Net, encode

NET_CONFIG = dict(rows=SPEC.rows, cols=SPEC.cols, channels=64, blocks=4)
net = Net(**NET_CONFIG).to(device)
n_params = sum(p.numel() for p in net.parameters())
print(f"{n_params:,} parameters (limit {MAX_PARAMS:,})")

b = Board(SPEC)
for col in [SPEC.cols // 2, SPEC.cols // 2, SPEC.cols // 2 - 1]:
    b.play(col)
with torch.no_grad():
    logits, value = net(encode([b]).to(device))
print("input", tuple(encode([b]).shape), " logits", tuple(logits.shape), " value", round(value.item(), 3))


class NetPlayer:
    """Plays the legal column with the highest logit. This is what the template hands in."""

    def __init__(self, net, name="net"):
        self.net, self.name = net, name

    @torch.no_grad()
    def choose(self, board):
        self.net.eval()
        logits, _ = self.net(encode([board]).to(device))
        logits = logits[0].float().cpu().numpy()
        logits[~board.legal_mask()] = -np.inf
        return int(np.argmax(logits))

## 4. Training: PPO against the bots

Many games run at once. In each game your network plays one color, and a bot
picked at random (with the weights in `Config.opponents`) plays the other. The
reward is +1 for a win, -1 for a loss, 0 for a draw, at the end of the game.
Illegal columns are masked out of the policy.

The defaults train for a few minutes. They learn to beat `random` and `easy`; the
stronger bots need much more. Section 6 lists ideas.

In [ ]:
@dataclasses.dataclass
class Config:
    updates: int = 60                 # raise this a lot for real training
    n_games: int = 64                 # games played in parallel
    rollout_len: int = 16             # moves per game per update
    gamma: float = 0.99
    gae_lambda: float = 0.95
    lr: float = 3e-4
    clip_eps: float = 0.2
    epochs: int = 4
    minibatch: int = 256
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    opponents: dict = dataclasses.field(default_factory=lambda: {"random": 0.4, "easy": 0.4, "medium": 0.15, "hard": 0.05})
    seed: int = 0


class Game:
    """One game: your net plays `side`, a bot plays the other color."""

    def __init__(self, rng, cfg):
        names = list(cfg.opponents)
        w = np.array([cfg.opponents[n] for n in names], dtype=float)
        self.opp = names[rng.choice(len(names), p=w / w.sum())]
        self.side = int(rng.integers(1, 3))
        self.board = Board(SPEC)
        self.bot_moves()

    def bot_moves(self):
        while not self.board.done and self.board.player != self.side:
            self.board.play(bots[self.opp].choose(self.board.copy()))


def collect(net, games, cfg, rng, tally):
    """cfg.rollout_len moves in every game. Returns the rollout as tensors."""
    buf = {k: [] for k in ("obs", "mask", "act", "logp", "val", "rew", "done")}
    for _ in range(cfg.rollout_len):
        boards = [g.board for g in games]
        obs = encode(boards).to(device)
        mask = torch.from_numpy(np.stack([b.legal_mask() for b in boards])).to(device)
        with torch.no_grad():
            logits, values = net(obs)
            dist = Categorical(logits=logits.masked_fill(~mask, float("-inf")))
            act = dist.sample()
        rew = np.zeros(len(games), dtype=np.float32)
        done = np.zeros(len(games), dtype=np.float32)
        for i, (g, a) in enumerate(zip(games, act.tolist())):
            g.board.play(a)
            g.bot_moves()
            if g.board.done:
                w = g.board.winner
                rew[i] = 0.0 if w == 0 else (1.0 if w == g.side else -1.0)
                done[i] = 1.0
                tally.setdefault(g.opp, np.zeros(3, int))[0 if rew[i] > 0 else 1 if rew[i] == 0 else 2] += 1
                games[i] = Game(rng, cfg)
        for k, v in (("obs", obs), ("mask", mask), ("act", act), ("logp", dist.log_prob(act)), ("val", values),
                     ("rew", torch.from_numpy(rew).to(device)), ("done", torch.from_numpy(done).to(device))):
            buf[k].append(v)
    with torch.no_grad():
        _, last_value = net(encode([g.board for g in games]).to(device))
    return {k: torch.stack(v) for k, v in buf.items()}, last_value


def gae(rew, val, done, last_value, gamma, lam):
    adv = torch.zeros_like(rew)
    running = torch.zeros_like(last_value)
    for t in reversed(range(rew.shape[0])):
        next_v = last_value if t == rew.shape[0] - 1 else val[t + 1]
        nd = 1.0 - done[t]
        running = rew[t] + gamma * next_v * nd - val[t] + gamma * lam * nd * running
        adv[t] = running
    return adv, adv + val


def ppo_update(net, opt, roll, adv, ret, cfg):
    obs, mask, act, old_logp = (roll[k].flatten(0, 1) for k in ("obs", "mask", "act", "logp"))
    adv, ret = adv.flatten(), ret.flatten()
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    net.train()
    for _ in range(cfg.epochs):
        perm = torch.randperm(len(obs), device=device)
        for s in range(0, len(obs), cfg.minibatch):
            i = perm[s:s + cfg.minibatch]
            logits, v = net(obs[i])
            dist = Categorical(logits=logits.masked_fill(~mask[i], float("-inf")))
            ratio = torch.exp(dist.log_prob(act[i]) - old_logp[i])
            pg = -torch.min(ratio * adv[i], ratio.clamp(1 - cfg.clip_eps, 1 + cfg.clip_eps) * adv[i]).mean()
            loss = pg + cfg.vf_coef * ((v - ret[i]) ** 2).mean() - cfg.ent_coef * dist.entropy().mean()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.max_grad_norm)
            opt.step()
    net.eval()


def train(cfg, net):
    rng = np.random.default_rng(cfg.seed)
    torch.manual_seed(cfg.seed)
    opt = torch.optim.Adam(net.parameters(), lr=cfg.lr)
    games = [Game(rng, cfg) for _ in range(cfg.n_games)]
    tally, t0 = {}, time.time()
    for update in range(1, cfg.updates + 1):
        net.eval()
        roll, last_value = collect(net, games, cfg, rng, tally)
        adv, ret = gae(roll["rew"], roll["val"], roll["done"], last_value, cfg.gamma, cfg.gae_lambda)
        ppo_update(net, opt, roll, adv, ret, cfg)
        if update % 10 == 0:
            line = "  ".join(f"{n} {t[0]}/{t[1]}/{t[2]}" for n, t in sorted(tally.items()))
            print(f"update {update:4d}  {time.time() - t0:5.0f}s  wins/draws/losses: {line}", flush=True)
            tally.clear()
    return net


cfg = Config()
net = train(cfg, Net(**NET_CONFIG).to(device))

## 5. How good is it?

`evaluate` plays your network against each bot: every opening twice, once with
each color, the way the grader plays. Then watch a game: the next cell writes an
HTML page that replays games move by move.

In [ ]:
def evaluate(player, n_openings=10, seed=2):
    openings = make_openings(SPEC, n_openings, seed=seed)
    scores = {}
    for name, bot in bots.items():
        r = play_match(player, bot, SPEC, openings)
        scores[name] = r.score
        print(r)
    print(f"mean score {np.mean(list(scores.values())):.3f}")
    return scores


me = NetPlayer(net, "you")
scores = evaluate(me)

In [ ]:
recs = [record_game(SPEC, me, bots["medium"], op) for op in make_openings(SPEC, 2, seed=3)]
recs += [record_game(SPEC, bots["medium"], me, op) for op in make_openings(SPEC, 2, seed=3)]
write_html(recs, "replay_a.html", "Your net vs medium")
print("wrote replay_a.html: open it in a browser")

## 6. Make it better

The template is a start, not a good bot. Ideas, roughly from cheap to expensive:

* **Train longer, against harder bots.** Raise `updates`, and move the
  `opponents` weights toward `medium` and `hard` as your net improves (a
  curriculum). Against strong bots from the start, a new net almost never wins,
  so it gets almost no learning signal.
* **Self-play.** Add frozen copies of your own net as opponents (see
  notebook b). The held-out bots are not the four you have.
* **Learn from games.** Record many games between the bots and train the
  network to copy the moves of the side that **won** (supervised learning).
  Then improve it with RL.
* **A bigger network.** You have up to 200 million parameters, but a move must
  take at most 1 second on one CPU core. Measure before you grow it.
* **Search at play time.** `agent.py` may run a search (for example MCTS) that
  calls your network, within the time limit.
* **Use the symmetry.** A position and its mirror image (columns reversed) are
  worth the same.

## 7. Hand in

The next cell writes the folder `submission_a/` and checks it like the grader
does. Hand in that folder (zip it). If you change `agent.py` (for example, to
add a search), edit `AGENT_PY` below or the file in the folder, and run
`check` again.

In [ ]:
def save_submission(net, folder):
    """Write a submission folder: agent.py, model.py, config.json, weights.pt."""
    folder = pathlib.Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    shutil.copy("model_a.py", folder / "model.py")
    (folder / "agent.py").write_text(AGENT_PY)
    (folder / "config.json").write_text(json.dumps(NET_CONFIG))
    torch.save({k: v.detach().cpu() for k, v in net.state_dict().items()}, folder / "weights.pt")
    return folder


AGENT_PY = '"""Submission: the network plays the legal column with the highest logit.\n\nYou may change this file too (for example, to add a search that calls your\nnetwork), as long as agent.net holds all the parameters and a move takes at\nmost 1 second on one CPU core.\n"""\nimport json\nimport pathlib\n\nimport numpy as np\nimport torch\n\nfrom model import Net, encode\n\n\nclass Agent:\n    def __init__(self, folder):\n        config = json.loads((folder / "config.json").read_text())\n        self.net = Net(**config)\n        self.net.load_state_dict(torch.load(folder / "weights.pt", map_location="cpu", weights_only=True))\n        self.net.eval()\n\n    @torch.no_grad()\n    def choose(self, board):\n        logits, _ = self.net(encode([board]))\n        logits = logits[0].numpy()\n        logits[~board.legal_mask()] = -np.inf\n        return int(np.argmax(logits))\n\n\ndef make_agent(folder):\n    return Agent(pathlib.Path(folder))\n'

save_submission(net, "submission_a")
report = check("submission_a", SPEC)          # the same checks as the grader: parameters, time per move, legal moves